# Terrain slope and aspect with Horn's 3×3 operator

This lab estimates local slope and downslope aspect from a regular elevation grid.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / 'implementations').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'implementations/python/terrain-derivatives'))
from terrain_derivatives import horn_slope_aspect


## 1. An east-rising plane

A plane that rises one elevation unit for each horizontal unit to the east has a 45° slope and a downslope aspect of 270° (west).

In [ ]:
dem = np.tile(np.arange(9, dtype=float), (9, 1))
result = horn_slope_aspect(dem)
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
layers = [dem, result['slope_degrees'], result['aspect_degrees']]
titles = ['Elevation', 'Slope (degrees)', 'Downslope aspect (degrees)']
for ax, layer, title in zip(axes, layers, titles):
    im = ax.imshow(layer, cmap='viridis'); ax.set_title(title); fig.colorbar(im, ax=ax, shrink=.75)
plt.tight_layout(); plt.show()
print('Centre slope:', result['slope_degrees'][4, 4])
print('Centre aspect:', result['aspect_degrees'][4, 4])


## 2. Local hillshade

Hillshade estimates local illumination from terrain slope/aspect and a chosen sun position. It does not cast shadows from distant terrain.

In [ ]:
from hillshade import hillshade_from_slope_aspect
shade = hillshade_from_slope_aspect(result['slope_degrees'], result['aspect_degrees'], sun_azimuth_degrees=315, sun_altitude_degrees=45)
plt.figure(figsize=(5, 4)); plt.imshow(shade, cmap='gray', vmin=0, vmax=1); plt.title('Local hillshade'); plt.colorbar(label='Normalized illumination'); plt.show()


## 3. Change horizontal spacing

Keep the elevation difference but double the east-west cell spacing. The gradient becomes half as steep.

In [ ]:
wide = horn_slope_aspect(dem, cell_size_x=2, cell_size_y=1)
print('Square-cell slope:', result['slope_degrees'][4, 4])
print('Wider-cell slope:', wide['slope_degrees'][4, 4])


## 4. Flats and missing elevation

Aspect is undefined on a flat. The strict validity policy masks every output window containing NoData, and the outer border.

In [ ]:
flat = horn_slope_aspect(np.ones((7, 7)))
with_gap = dem.copy(); with_gap[3, 3] = np.nan
gap = horn_slope_aspect(with_gap)
print('Flat slope/aspect:', flat['slope_degrees'][3, 3], flat['aspect_degrees'][3, 3])
print(np.isfinite(gap['slope_degrees']).astype(int))


## Investigate

1. Change the plane so it rises northward. What aspect do you expect?
2. Change the vertical conversion factor. Why does aspect remain the same?
3. What happens to aspect when slope approaches zero?
4. Why does strict NoData masking reduce valid coverage around a gap?
5. Compare Horn with a central-difference method and state the differences in support and edge behavior.

## 5. Topographic wetness index

TWI combines specific catchment area and slope. Here the area surface is illustrative; in a real workflow it must be derived from a flow-routing model and converted to specific area in metres.

In [ ]:
from terrain_indices import topographic_wetness_index
specific_area = np.tile(np.linspace(2, 20, 9), (9, 1))
twi = topographic_wetness_index(specific_area, result['slope_degrees'])
plt.figure(figsize=(5, 4)); plt.imshow(twi, cmap='viridis'); plt.title('Illustrative TWI'); plt.colorbar(label='Index value'); plt.show()
print('TWI is an index, not a soil-moisture observation.')
